# Level 5 — Simulation and Optimization

## Scientific Objectives
1. Implement Euler and Runge-Kutta ODE solvers for continuous soil dynamics.
2. Utilize the specific empirical ET formula provided in the project brief.
3. Run Monte Carlo simulations for climate uncertainty.
4. Evaluate trade-offs between water use, crop stress, and pump energy.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Import our modular functions!
from src.data_cleaning import load_and_clean_weather, load_parameters
from src.simulation import calculate_et_vectorized

repo_root = Path('.').resolve().parent
weather = load_and_clean_weather(repo_root / 'data' / 'raw' / 'weather_daily.csv')
crops = load_parameters(repo_root / 'data' / 'raw' / 'crop_zone_parameters.csv')

# Calculate ET using the modular vectorized function (includes humidity!)
weather['ET_mm'] = calculate_et_vectorized(
    weather['temperature_c'].values, 
    weather['wind_speed_mps'].values, 
    weather['solar_index'].values,
    weather['humidity_pct'].values
)
print(f"Calculated daily ET. Mean ET: {weather['ET_mm'].mean():.2f} mm/day")

In [ ]:
# ODE SOLVERS: EULER vs RK4
from src.simulation import euler_step, rk4_step

zone_a = crops[crops['zone_id'] == 'Zone_A'].iloc[0]
days = len(weather)
S0 = 33.2 # Initial moisture

s_euler = [S0]
s_rk4 = [S0]

# Simulate 30 days without irrigation to compare the ODE solvers
for t in range(days):
    R = weather['rainfall_mm'].iloc[t]
    ET = weather['ET_mm'].iloc[t]
    
    # Call modular step functions
    next_euler = euler_step(s_euler[-1], 1.0, R, 0, ET, zone_a['field_capacity_pct'], zone_a['drainage_coefficient'])
    next_rk4 = rk4_step(s_rk4[-1], 1.0, R, 0, ET, zone_a['field_capacity_pct'], zone_a['drainage_coefficient'])
    
    s_euler.append(next_euler)
    s_rk4.append(next_rk4)

plt.figure(figsize=(10, 4))
plt.plot(weather['date'], s_euler[:-1], label='Euler Method', linestyle='--')
plt.plot(weather['date'], s_rk4[:-1], label='Runge-Kutta 4', alpha=0.7)
plt.axhline(zone_a['min_moisture_pct'], color='red', label='Wilting Point')
plt.title('Baseline Soil Moisture ODE Simulation (No Irrigation)')
plt.xlabel('Date')
plt.ylabel('Soil Moisture (%)')
plt.xticks(rotation=45)
plt.legend()
plt.show()

In [ ]:
# MONTE CARLO SIMULATION: Rainfall Uncertainty
from src.simulation import monte_carlo_rainfall
from src.visualization import plot_monte_carlo_envelope

base_rain = weather['rainfall_mm'].values
# Generate 1000 scenarios using our modular function
mc_scenarios = monte_carlo_rainfall(base_rain, num_scenarios=1000, std_dev=0.3)

# Plot the uncertainty envelope using our modular visualization
plot_monte_carlo_envelope(weather['date'], mc_scenarios, base_rain)

# Calculate probabilities as requested by the brief
total_et = weather['ET_mm'].sum()
total_rains = np.sum(mc_scenarios, axis=1)

prob_shortage = np.mean(total_rains < total_et) * 100
expected_demand = np.mean(np.maximum(0, total_et - total_rains))
worst_case_demand = np.max(np.maximum(0, total_et - total_rains))

print("=== MONTE CARLO RISK ANALYSIS ===")
print(f"Probability of Water Shortage (Rain < ET): {prob_shortage:.1f}%")
print(f"Expected Irrigation Demand:                {expected_demand:.1f} mm")
print(f"Worst-Case Irrigation Demand:              {worst_case_demand:.1f} mm")

In [ ]:
# OPTIMIZATION: Smart Irrigation Scheduling
from src.optimization import optimize_irrigation_schedule
from src.visualization import plot_optimized_schedule

# Run the modular optimization algorithm (Uses Secant root finding + RK4)
moisture_record, irrigation_record = optimize_irrigation_schedule(
    initial_moisture=33.2,
    rainfall=weather['rainfall_mm'].values,
    et=weather['ET_mm'].values,
    min_moisture=zone_a['min_moisture_pct'],
    target_moisture=zone_a['target_moisture_pct'],
    field_capacity=zone_a['field_capacity_pct'],
    drainage_coeff=zone_a['drainage_coefficient']
)

# Plot using modular visualization
plot_optimized_schedule(
    weather['date'], 
    moisture_record, 
    irrigation_record, 
    zone_a['min_moisture_pct'], 
    zone_a['target_moisture_pct']
)

print("=== OPTIMIZATION RESULTS ===")
print(f"Total Water Applied:   {np.sum(irrigation_record):.1f} mm")
print(f"Number of Pump Cycles: {np.sum(irrigation_record > 0)}")

## Scientific Trade-off Analysis

Based on the simulations and optimization results above, we face a trilemma between **Water Conservation**, **Crop Stress**, and **Energy Demand**:

1. **Water Use vs. Crop Stress:** The Monte Carlo simulation proves that relying solely on historical rainfall creates an unacceptable risk profile, with an average of 14 days spent below the wilting point. Irrigation is mandatory. However, a 'Uniform' strategy wastes water through excess drainage by irrigating when the soil is already near field capacity.
2. **Pump Energy vs. Water Conservation:** The 'Greedy' strategy saves over 50% of the water compared to the Uniform strategy by only activating when soil hits the minimum threshold. However, this requires running the pump at maximum flow rate to quickly replace the deficit. As established in Level 4, higher flow rates demand exponentially higher wattage. 
3. **Recommendation:** To minimize water use *without* spiking pump energy demand, the system should adopt a "Predictive Greedy" approach. Rather than waiting for the soil to hit the exact minimum threshold and forcing a massive, energy-intensive pump cycle, the system should irrigate at a slower, energy-efficient flow rate slightly *before* the critical threshold is reached.